# GraphQL — Capture Without Executing via `singleton_class.class_eval`

**Pattern**: intercept `AppSchema.execute` so outgoing GraphQL mutations are captured before the schema runs them — same principle as the Karafka demo.

- **Phase 1 (capture)**: `execute` is intercepted — mutation recorded, schema never runs, DB untouched.
- **Phase 2 (passthrough)**: patch is off — mutation executes normally.

Rollback is intentionally out of scope: like Kafka messages, GraphQL mutations typically trigger side effects in external systems that cannot be cleanly reversed here.

## Setup — schema with a mutation

> **Full reset only:** restart the kernel and re-run all cells — GraphQL schema classes cannot be safely redefined mid-session.

In [1]:
require "active_record"
require "graphql"

Object.send(:remove_const, :DB_PATH) if defined?(DB_PATH)
DB_PATH = "/lab/ruby/notebooks/graphql_demo.db"

ActiveRecord::Base.establish_connection(adapter: "sqlite3", database: DB_PATH)
ActiveRecord::Schema.verbose = false
ActiveRecord::Schema.define do
  create_table :users, force: :cascade do |t|
    t.string  :name
    t.string  :email
    t.integer :age
    t.timestamps
  end
end

class User < ActiveRecord::Base
  self.table_name = "users"
end

User.create!(name: "Alice", email: "alice@example.com", age: 30)
User.create!(name: "Bob",   email: "bob@example.com",   age: 25)

module Types
  class UserType < GraphQL::Schema::Object
    field :id,    ID,      null: false
    field :name,  String,  null: true
    field :age,   Integer, null: true
  end

  class QueryType < GraphQL::Schema::Object
    field :users, [UserType], null: false
    def users = User.all
  end

  class UpdateUserAgeMutation < GraphQL::Schema::Mutation
    argument :id,  ID,      required: true
    argument :age, Integer, required: true
    field :user, UserType, null: true

    def resolve(id:, age:)
      user = User.find(id)
      user.update!(age: age)
      { user: user.reload }
    end
  end

  class MutationType < GraphQL::Schema::Object
    field :update_user_age, mutation: UpdateUserAgeMutation
  end
end

class AppSchema < GraphQL::Schema
  query    Types::QueryType
  mutation Types::MutationType
end

User.all.each { |u| puts "  #{u.name}: age=#{u.age}" }

  Alice: age=30
  Bob: age=25


[#<#<Class:0x00007f9ed1706030>::User id: 1, name: "Alice", email: "alice@example.com", age: 30, created_at: "2026-06-01 07:16:27.611208000 +0000", updated_at: "2026-06-01 07:16:27.611208000 +0000">, #<#<Class:0x00007f9ed1706030>::User id: 2, name: "Bob", email: "bob@example.com", age: 25, created_at: "2026-06-01 07:16:27.612879000 +0000", updated_at: "2026-06-01 07:16:27.612879000 +0000">]

## CaptureContext

In [2]:
module CaptureContext
  @capturing = false
  @mutations = []

  def self.start!     = (@capturing = true; @mutations = [])
  def self.stop!      = (@capturing = false)
  def self.capturing? = @capturing

  def self.add(query:, variables: nil)
    @mutations << { query: query, variables: variables }
  end

  def self.mutations = @mutations.dup
end

puts "CaptureContext ready"

CaptureContext ready


## The patch — `AppSchema.singleton_class.class_eval`

`execute` is a **class method** on `AppSchema` (called as `AppSchema.execute(...)`) — so we open the **singleton class** to patch it.

This intercepts **all** GraphQL operations through the schema automatically. No per-mutation patches needed.

| Patch | Method | Level |
|---|---|---|
| `ActiveRecord::Base.class_eval` | `before_save` (instance) | framework base class |
| `AppSchema.singleton_class.class_eval` | `execute` (class method) | app schema class |
| `WaterDrop::Producer.class_eval` | `produce_sync` (instance) | framework class |

In [3]:
# NOTE: run this cell only once per kernel session.
# Re-running alias_method points :original_execute at the already-patched method,
# causing infinite recursion. To reset, restart the kernel and re-run all cells.
AppSchema.singleton_class.class_eval do
  alias_method :original_execute, :execute

  def execute(query_str = nil, **kwargs)
    if CaptureContext.capturing?
      CaptureContext.add(query: query_str, variables: kwargs[:variables])
      return {}
    end

    original_execute(query_str, **kwargs)
  end
end

puts "Patch applied -- AppSchema.singleton_class.class_eval execute wrapped"

Patch applied -- AppSchema.singleton_class.class_eval execute wrapped


## Demo mutation

In [4]:
MUTATION = 'mutation { updateUserAge(id: "1", age: 99) { user { name age } } }'

"mutation { updateUserAge(id: \"1\", age: 99) { user { name age } } }"

## Phase 1 — Capture Mode

`AppSchema.execute` is called but intercepted. The mutation string is recorded; the schema never runs.

In [ ]:
CaptureContext.start!

AppSchema.execute(MUTATION)

puts "Intercepted #{CaptureContext.mutations.size} mutation(s) -- schema never executed:"
CaptureContext.mutations.each_with_index do |m, i|
  puts "  [#{i}] #{m[:query].strip}"
end
puts "DB -> Alice: age=#{User.find_by(name: 'Alice').age}   (unchanged)"

## Phase 2 — Passthrough

Capture is off. `execute` calls through to `original_execute` — the mutation runs normally.

In [ ]:
CaptureContext.stop!

result = AppSchema.execute(MUTATION)
data   = result['data']&.dig('updateUserAge', 'user')
puts "Result -> #{data.inspect}"
puts "DB -> Alice: age=#{User.find_by(name: 'Alice').age}"